**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**05 | Temporal Learning and Advanced Multimodal Fusion**

![Temporal Learning and Advanced Multimodal Fusion](../../images/05_temporal_and_advanced_fusion.svg)

**Objective:** compare compact EEG-only GRU, gaze-only GRU and
       gated EEG+gaze GRU models using matched six-bin temporal sequences.
       **Input:** Stage-03 EEG/gaze sequences and fixed Train/Validation
       roles. **Evaluation:** record-level window predictions on Validation;
       the independent Test is not accessed for model choice. Capacity is
       intentionally constrained because the independent participant count
       is small even when thousands of windows are available.

**00. Colab Repository Setup**

In [ ]:
# ==========================00. Colab Repository Setup=========================

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
PROJECT = "eeg_eye_tracking_fusion"
repo_root = Path('/content/Eye-Tracking-Neurocognitive-Intelligence')
if not Path('/content').is_dir():
    raise RuntimeError('These notebooks target Google Colab.')
if not (repo_root / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                    str(repo_root)], check=True)
work = repo_root / PROJECT
output_dir = work / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path('/content/zuco2_source_cache')
raw_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(work / 'code' / 'src'))
print('Repository:', repo_root)
print('Project:', work)
print('Outputs:', output_dir)
print('Source cache (temporary):', raw_dir)


**01. Imports and Research Environment**

In [ ]:
# =====================01. Imports and Research Environment====================

import importlib.util
import json
import random
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec('mne') is None:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'mne>=1.6,<2'], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 17
random.seed(SEED)
np.random.seed(SEED)
print('Python:', sys.version.split()[0])
print('Free disk (GiB):', round(
    __import__('shutil').disk_usage('/content').free / 2**30, 2))
print('NumPy:', np.__version__, 'pandas:', pd.__version__)


**02. Load Sequences and Fit Train-Only Scaling**

Missing-value replacement, centring and scaling parameters
          are fitted on Train windows and saved for identical application
          to Validation and, later, Test. Participant IDs are not features.

In [ ]:
# ================02. Load Sequences and Fit Train-Only Scaling================

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from models import (NEURAL_ORDER, CLASS_TO_INDEX, metrics,
                    record_predictions, torch_network,
                    predict_neural)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sequences = np.load(output_dir / '03_temporal_sequences.npz',
                    allow_pickle=False)
table = pd.read_csv(output_dir / '03_features.csv')
assert np.array_equal(sequences['window_id'],
                      table.window_id.astype(str).to_numpy())
y = table.task.map(CLASS_TO_INDEX).to_numpy(dtype=int)
roles = table.split.to_numpy()
idx_train, idx_val = np.where(roles == 'train')[0], np.where(
    roles == 'validation')[0]
if min(len(idx_train), len(idx_val)) < 1:
    raise ValueError('Train or Validation is empty')

def fit_scaler(original, training_rows):
    train = original[training_rows].reshape(-1, original.shape[-1])
    med = np.nanmedian(train, axis=0)
    med = np.where(np.isfinite(med), med, 0.)
    filled = np.where(np.isfinite(train), train, med)
    mean = filled.mean(axis=0)
    std = filled.std(axis=0)
    return med.astype('float32'), mean.astype('float32'), np.maximum(
        std, 1e-6).astype('float32')

def transform(original, median, mean, std):
    filled = np.where(np.isfinite(original), original, median)
    return ((filled - mean) / std).astype('float32')

eeg_med, eeg_mean, eeg_std = fit_scaler(sequences['eeg'], idx_train)
gaze_med, gaze_mean, gaze_std = fit_scaler(sequences['gaze'], idx_train)
eeg = transform(sequences['eeg'], eeg_med, eeg_mean, eeg_std)
gaze = transform(sequences['gaze'], gaze_med, gaze_mean, gaze_std)
np.savez_compressed(output_dir / '05_scaling.npz',
                    eeg_median=eeg_med, eeg_mean=eeg_mean,
                    eeg_std=eeg_std, gaze_median=gaze_med,
                    gaze_mean=gaze_mean, gaze_std=gaze_std)
print('Device:', device, 'Train windows:', len(idx_train),
      'Validation windows:', len(idx_val))


**03. Define Compact Architectures and Fixed Training Settings**

Three candidates share the same Train/Validation windows,
          optimiser and 16-unit hidden representation. The gated architecture
          combines learned EEG and gaze embeddings; gate values are not
          causal explanations of the predicted task.

In [ ]:
# =========03. Define Compact Architectures and Fixed Training Settings========

HIDDEN = 16
EPOCHS = 15
PATIENCE = 4
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-3
train_set = TensorDataset(
    torch.from_numpy(eeg[idx_train]),
    torch.from_numpy(gaze[idx_train]),
    torch.from_numpy(y[idx_train]).long())
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator)
print('Six-step sequence shape:', eeg.shape,
      'Candidate models:', NEURAL_ORDER)


**04. Train EEG-Only, Gaze-Only and Gated Fusion GRUs**

Choose checkpoints using Validation fixed-label Macro F1,
          then Balanced Accuracy and lower Validation cross-entropy.
          No model is trained on or selected by Test observations.

In [ ]:
# =============04. Train EEG-Only, Gaze-Only and Gated Fusion GRUs=============

histories, validation_rows, prediction_frames = [], [], []
fitted = {}
for name in NEURAL_ORDER:
    model = torch_network(name, eeg.shape[-1], gaze.shape[-1],
                          hidden=HIDDEN).to(device)
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    best_key, best_weights, best_epoch = None, None, None
    stale = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for a, b, truth in train_loader:
            a, b, truth = a.to(device), b.to(device), truth.to(device)
            optimiser.zero_grad()
            loss = criterion(model(a, b), truth)
            loss.backward()
            optimiser.step()
            losses.append(float(loss.detach().cpu()))
        proba = predict_neural(model, eeg[idx_val], gaze[idx_val],
                               device=device)
        scores = metrics(y[idx_val], proba.argmax(axis=1))
        val_loss = float(-np.log(np.clip(
            proba[np.arange(len(idx_val)), y[idx_val]], 1e-8, 1
        )).mean())
        key = (scores['macro_f1'], scores['balanced_accuracy'],
               -val_loss)
        histories.append({'model': name, 'epoch': epoch,
                          'train_loss': float(np.mean(losses)),
                          'validation_loss': val_loss, **scores})
        if best_key is None or key > best_key:
            best_key = key
            best_epoch = epoch
            best_weights = {k: v.detach().cpu().clone()
                            for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_weights)
    fitted[name] = model.cpu()
    result = predict_neural(model, eeg[idx_val], gaze[idx_val])
    scores = metrics(y[idx_val], result.argmax(axis=1))
    validation_rows.append({
        'model': name, 'partition': 'validation',
        'n_windows': len(idx_val),
        'n_people': table.iloc[idx_val].participant_id.nunique(),
        'parameters': sum(p.numel() for p in model.parameters()),
        'best_epoch': best_epoch, **scores})
    prediction_frames.append(record_predictions(
        table.iloc[idx_val], name, result))
    torch.save({'state_dict': best_weights, 'model': name,
                'hidden': HIDDEN, 'eeg_features': eeg.shape[-1],
                'gaze_features': gaze.shape[-1],
                'best_epoch': best_epoch},
               output_dir / f'05_{name}.pt')
    print(name, 'best epoch', best_epoch,
          'Validation F1', round(scores['macro_f1'], 4))


**05. Review Learning Curves and Validation Ranking**

Learning curves are development evidence only. A complex
          fusion architecture need not outperform its EEG-only and gaze-only
          controls, particularly with few independent participants.

In [ ]:
# ==============05. Review Learning Curves and Validation Ranking==============

history = pd.DataFrame(histories)
neural_metrics = pd.DataFrame(validation_rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, frame in history.groupby('model'):
    axes[0].plot(frame.epoch, frame.train_loss, label=name)
    axes[1].plot(frame.epoch, frame.macro_f1, label=name)
axes[0].set(title='Training cross-entropy', xlabel='Epoch',
            ylabel='Loss')
axes[1].set(title='Independent-person Validation',
            xlabel='Epoch', ylabel='Macro F1', ylim=(0, 1.05))
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / '05_learning_curves.png', dpi=170)
plt.show()
display(neural_metrics.sort_values('macro_f1', ascending=False))


**06. Verify and Save Stage-05 Validation Predictions**

All three models must cover the same Validation source IDs,
          with valid probabilities and correct task labels. The fixed model
          snapshots are needed by Notebook 06; no Test metrics appear here.

In [ ]:
# =============06. Verify and Save Stage-05 Validation Predictions=============

pred = pd.concat(prediction_frames, ignore_index=True)
for name, piece in pred.groupby('model'):
    assert piece.window_id.is_unique
    assert set(piece.window_id) == set(table.iloc[idx_val].window_id)
    assert np.allclose(piece[['p_nr', 'p_tsr']].sum(axis=1), 1.)
pred.to_csv(output_dir / '05_validation_predictions.csv',
            index=False)
neural_metrics.to_csv(output_dir / '05_validation_metrics.csv',
                      index=False)
history.to_csv(output_dir / '05_training_history.csv', index=False)
print('Neural Validation rows:', len(pred))


**07. Interpretation and Next Dependency**

Notebook 06 applies the prespecified ranking across all ten
          candidates, then evaluates frozen models once on held-out Test.
          A small development cohort can check the implementation but does
          not provide strong population-level performance estimates.

In [ ]:
# ====================07. Interpretation and Next Dependency===================

assert set(neural_metrics.model) == set(NEURAL_ORDER)
print('Next: 06_robustness_and_final_evaluation.ipynb')
